In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
accepted = pd.read_csv("/Users/irsyad/Desktop/Portfolio/credit-risk-lending-club/data/raw/accepted_2007_to_2018Q4.csv", nrows=100000,low_memory=False)
print(accepted.shape)

(100000, 151)


In [3]:
accepted.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
accepted.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [5]:
accepted.info(verbose = True,show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 151 columns):
 #    Column                                      Non-Null Count   Dtype  
---   ------                                      --------------   -----  
 0    id                                          100000 non-null  int64  
 1    member_id                                   0 non-null       float64
 2    loan_amnt                                   100000 non-null  float64
 3    funded_amnt                                 100000 non-null  float64
 4    funded_amnt_inv                             100000 non-null  float64
 5    term                                        100000 non-null  str    
 6    int_rate                                    100000 non-null  float64
 7    installment                                 100000 non-null  float64
 8    grade                                       100000 non-null  str    
 9    sub_grade                                   100000 non-null  str    


In [6]:
# Check for null pecentages
null_pct = accepted.isnull().sum() / len(accepted) * 100
null_pct = null_pct[null_pct>0].sort_values(ascending=False)
print(null_pct)

member_id                   100.000
revol_bal_joint             100.000
sec_app_fico_range_high     100.000
sec_app_earliest_cr_line    100.000
sec_app_inq_last_6mths      100.000
                             ...   
last_pymnt_d                  0.071
revol_util                    0.037
last_credit_pull_d            0.003
dti                           0.002
num_rev_accts                 0.001
Length: 73, dtype: float64


In [7]:
high_null = null_pct[null_pct >= 70].index.tolist()
mid_null = null_pct[(null_pct>=30) & (null_pct<70)].index.tolist()
low_null = null_pct[(null_pct > 0) & (null_pct < 30)].index.tolist()

print(f"70%+ null (likely drop): {len(high_null)} columns")
print(f"30-70% null (investigate): {len(mid_null)} columns")
print(f"Under 30% null (likely keep): {len(low_null)} columns")

70%+ null (likely drop): 56 columns
30-70% null (investigate): 2 columns
Under 30% null (likely keep): 15 columns


In [8]:
print(mid_null)

['mths_since_recent_revol_delinq', 'mths_since_last_delinq']


In [9]:
accepted.drop(columns=high_null,inplace = True)

In [10]:
# fill -1 for null values as they have not been delinquent before
accepted["mths_since_last_delinq"] = accepted["mths_since_last_delinq"].fillna(-1)
accepted["mths_since_recent_revol_delinq"] = accepted["mths_since_recent_revol_delinq"].fillna(-1)

In [11]:
accepted.shape

(100000, 95)

In [23]:
# Remaining columns with null values
accepted.isnull().sum()[accepted.isnull().sum() > 0].sort_values(ascending=False)

last_pymnt_d          71
last_credit_pull_d     3
dtype: int64

In [13]:
accepted['mths_since_recent_inq'] = accepted['mths_since_recent_inq'].fillna(-1)
accepted['num_tl_120dpd_2m'] = accepted['num_tl_120dpd_2m'].fillna(0)

# Can't impute employment meaningfully, flag as unknown
accepted['emp_title'] = accepted['emp_title'].fillna('Unknown')
accepted['emp_length'] = accepted['emp_length'].fillna('Unknown')

# Small-medium nulls, median is safest for skewed financial data
cols_fill_median = ['bc_open_to_buy', 'bc_util', 'mo_sin_old_il_acct',
                    'mths_since_recent_bc', 'num_rev_accts', 'percent_bc_gt_75',
                    'revol_util', 'dti']

for col in cols_fill_median:
    accepted[col] = accepted[col].fillna(accepted[col].median())

accepted['title'] = accepted['title'].fillna('Unknown')



In [14]:
pd.options.display.max_rows = 1000

In [15]:
date_cols = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']

for cols in date_cols:
    accepted[cols] = pd.to_datetime(accepted[cols], format = "%b-%Y", errors='coerce')

In [16]:
accepted[date_cols].isnull().sum()

issue_d                0
earliest_cr_line       0
last_pymnt_d          71
last_credit_pull_d     3
dtype: int64

In [18]:
print(accepted.isnull().sum().sum())

74


In [20]:
print(accepted.shape)

(100000, 95)


In [21]:
accepted.to_csv('/Users/irsyad/Desktop/Portfolio/credit-risk-lending-club/data/processed/accepted_cleaned.csv', index=False)